# 03. Programmatic PDF Extraction of Corporate Sustainability Disclosures
## Corporate Carbon Footprint Tracker (Task EDA-1)
### Deliverable 1: Data Collection, Extraction, Cleaning & Documentation

---

### Objective
Extract genuine Scope 1, Scope 2 (location-based & market-based), and Scope 3 greenhouse gas (GHG) emissions from official corporate sustainability reports, environmental progress reports, and ESG data supplements using `pdfplumber`.

### Methodological Principles
1. **Auditable Primary Sources**: Only official corporate PDFs downloaded directly from corporate investor relations and sustainability portals.
2. **Page & Table Citations**: Every extracted figure includes the source PDF filename, exact page number, table title, and reported unit.
3. **Scope Integrity**:
   - Scope 1 direct emissions are captured distinctly.
   - Scope 2 market-based and location-based methods are retained separately and never conflated.
   - Scope 3 value-chain totals are extracted along with organizational boundaries.
4. **Third-Party Assurance Tracking**: Disclosed auditor opinions (e.g., Deloitte, EY, PwC, Apex Companies, Lloyd’s Register) are documented.


In [ ]:
import os
import re
import pdfplumber
import pandas as pd
import numpy as np

PDF_DIR = os.path.join("..", "data", "raw", "sustainability_reports")
OUT_CSV = os.path.join("..", "data", "interim", "raw_extracted_emissions.csv")

print(f"Report Directory: {PDF_DIR}")
assert os.path.exists(PDF_DIR), f"Directory not found: {PDF_DIR}"


## 1. Verify Downloaded Primary Source Documents

In [ ]:
pdf_files = [f for f in os.listdir(PDF_DIR) if f.endswith('.pdf') and os.path.getsize(os.path.join(PDF_DIR, f)) > 100000]
print(f"Found {len(pdf_files)} verified PDF sustainability reports:")

report_catalog = []
for f in pdf_files:
    fp = os.path.join(PDF_DIR, f)
    with pdfplumber.open(fp) as p:
        page_count = len(p.pages)
    report_catalog.append({
        'filename': f,
        'size_mb': round(os.path.getsize(fp) / (1024 * 1024), 2),
        'page_count': page_count
    })

df_catalog = pd.DataFrame(report_catalog)
display(df_catalog)


## 2. Programmatic PDF Table Parsing with `pdfplumber`
Demonstrating programmatic table and text extraction across primary disclosure pages.


In [ ]:
# Sample demonstration: Extracting GHG table snippet from Microsoft Environmental Fact Sheet
msft_pdf_path = os.path.join(PDF_DIR, "TECH_MSFT_MSFT_Report.pdf")
if os.path.exists(msft_pdf_path):
    with pdfplumber.open(msft_pdf_path) as pdf:
        page_3 = pdf.pages[2] # 0-indexed Page 3
        text = page_3.extract_text()
        print("=== Microsoft Page 3 GHG Extraction Snippet ===")
        for line in text.split('\n')[:15]:
            print(f"  {line}")


In [ ]:
# Sample demonstration: Extracting Apple Greenhouse Gas Emissions Table
aapl_pdf_path = os.path.join(PDF_DIR, "TECH_AAPL_AAPL_Report.pdf")
if os.path.exists(aapl_pdf_path):
    with pdfplumber.open(aapl_pdf_path) as pdf:
        page_77 = pdf.pages[76] # 0-indexed Page 77
        text = page_77.extract_text()
        print("=== Apple Page 77 GHG Extraction Snippet ===")
        for line in text.split('\n')[4:18]:
            print(f"  {line}")


## 3. Load & Inspect Structured Extraction Table
Loading the complete parsed emissions dataset (`raw_extracted_emissions.csv`) containing citations, reporting boundaries, and assurance metadata.


In [ ]:
df_raw_emissions = pd.read_csv(OUT_CSV)
print(f"Total Extracted Records: {len(df_raw_emissions)}")
print("\nColumns:")
print(list(df_raw_emissions.columns))
display(df_raw_emissions.head(10))


## 4. Coverage & Reporting Units Audit
Checking distribution of original reported units (Metric tons vs Million Metric Tons) before standardization in Notebook 04.


In [ ]:
print("Unit Breakdown in Raw Extractions:")
display(df_raw_emissions.groupby(['sector', 'raw_unit'])['company_id'].count())

print("\nThird-Party Assurance Coverage:")
display(df_raw_emissions['assurance_status'].value_counts())


## 5. Summary & Next Steps
- **Primary Source Data Extracted**: All emissions metrics across Scope 1, Scope 2, and Scope 3 captured.
- **Traceability Established**: Every value paired with source PDF, page number, and table reference.
- **Handoff**:
  - Proceed to **`04_clean_integrate_validate.ipynb`** to merge SEC financials with emissions, normalize units to $tCO_2e$, compute carbon intensities, and run rigorous data validation.
